# N117 · DP空间与转移优化

**目标：** 先证明依赖与候选支配再删除状态或转移。

**先修：** N098, N056, N031, N099。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 滚动数组；前缀和；单调队列；窗口DP；保留恢复路径的代价。

**配套讲解来源：** [同名逐章意见](../../comment/117_dp_optimization_basics.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

前几章的 DP 我们已经会写了，但朴素写法可能在"转移要扫一个窗口"或"要存整张表"上浪费大量时间/空间。这一章的主题是：**在动手删状态、删转移之前，先写清楚候选集合，再证明谁被谁"支配"，然后才敢删**。三个具体问题：

1. **约束子序列最大和（LC1425）：** 从数组里选一个子序列，要求相邻两个被选元素的下标距离不超过 k，子序列非空，求最大和。（子序列可以随时"重新开始"——它不要求从下标 0 出发。）
2. **跳跃游戏 VI（LC1696）：** 你站在下标 0，每步最多往前跳 k 格，必须一路跳到最后一个下标，得分是沿途踩到的数字之和，求最大得分。（不许重新开始，起点终点都定死。）
3. **逆序对数组（LC629）：** 1..n 的排列里，恰好有 k 个逆序对（i<j 但 p[i]>p[j]）的排列有多少个？答案对 10^9+7 取模。

**具体到数字的例子：** 跳跃游戏输入 `[1,-1,-2,4,-7,3]`、k=2，输出 **7**。走法是 0→1→3→5：踩到 1、−1、4、3，得分 7；每步跳跃距离是 1、2、2，都不超过 2。约束子序列输入 `[-3,-1,-2]`、k=2，输出 **−1**：全是负数也必须至少选一个，最好的选择是只选 −1。逆序对 `k_inverse_pairs(3,1)` 输出 **2**：1..3 的排列里恰有 1 个逆序对的是 (1,3,2) 和 (2,1,3) 两个。

## 2. 基础知识：先读懂这些词

- **候选集合：** 做 DP 转移时，dp[i] 依赖的所有前驱状态（这里是"下标落在窗口 [i−k, i−1] 里的那些 j"）。优化的第一步永远是把这个集合写清楚。
- **支配关系（dominance）：** 如果状态 j 比状态 i 更早过期（下标更小），且它的值也不比 i 大，那 i 就"支配"了 j——只要 i 还在窗口里，j 永远轮不到当最优；i 过期时 j 早就过期了。被支配的状态可以放心删掉。
- **单调队列（monotonic deque）：** 用双端队列维护滑动窗口最大值的结构。队列里存（下标, 值），保持"下标递增、值严格递减"：过期元素从队头弹出，被新元素支配的从队尾弹出。每个元素至多进队一次、出队一次，所以整个循环是均摊 O(1)。
- **窗口DP：** 转移形如 dp[i] = nums[i] + max（或 sum）dp[j]，j 在长度 k 的滑动窗口里。朴素写每个 i 都要扫窗口 O(k)，总 O(nk)；单调队列（max）或滑动和（sum）都能压到 O(n)。
- **滚动数组：** 转移只依赖"上一行"时，只保留两行（dp 和 nxt）轮流覆盖，把二维表的空间压成一维。逆序对计数用的就是它。
- **滑动和（滑动窗口求和）：** 求连续区间和时，每右移一格就"加进一个新的、减去一个滑出的"，避免每次重新求和。它和单调队列是窗口 DP 的两把刀：一个管 max，一个管 sum。
- **max(0, ·) 的语义：** 约束子序列允许"重新开始"——转移里 max(0, 窗口最大) 表示"要么接上前面的最优子序列，要么不带任何前缀、从当前元素单开一个新子序列"。跳跃游戏则**不能**加这个 0：起点必须是下标 0，链条不能断。
- **恢复路径的代价：** 队列优化只保留了"最优值"，丢掉了"从哪个前驱来"。要输出方案就得另外存一个前驱数组（O(n) 额外空间），这是省时间要付的账。

## 3. 思路推导：从小例子开始

### 3.1 两个窗口 max 题：单调队列

- **Step 1：** 写朴素转移。跳跃游戏：dp[i] = nums[i] + max(dp[i−k .. i−1])；约束子序列：s[i] = nums[i] + max(0, max s[i−k .. i−1])。区别就一个"要不要 0"。
- **Step 2：** 认出这是"滑动窗口最大值"，用单调队列把每个 max 从 O(k) 做到均摊 O(1)。
- **Step 3：** 队列不变量——按插入顺序，下标递增、值严格递减，队头永远是当前窗口的最大值。

**手算跳跃游戏 `[1,-1,-2,4,-7,3]`、k=2（“运行示例”部分的表格逐行记录了这个过程）：**

- i=0：dp=1，队列 [(0,1)]。
- i=1：队头下标 0 未过期，dp = −1 + 1 = 0；0 比队尾 1 小，入队 → [(0,1),(1,0)]。
- i=2：dp = −2 + 1 = −1，入队 → [(0,1),(1,0),(2,−1)]。
- i=3：队头 (0,·) 过期（0 < 3−2）弹出；dp = 4 + 0 = **4**；4 支配队尾 (1,0) 和 (2,−1)，全部弹出，入队 → [(3,4)]。
- i=4：dp = −7 + 4 = −3，入队 → [(3,4),(4,−3)]。
- i=5：队头下标 3 未过期（3 ≥ 5−2）；dp = 3 + 4 = **7**；(4,−3) 被 7 支配弹出，入队 → [(3,4),(5,7)]。返回 7。

**手算约束子序列 `[-3,-1,-2]`、k=2：** i=0：value = −3 + max(0, 空) = −3；i=1：value = −1 + max(0, −3) = −1 + 0 = −1（那个 max(0,·) 就是"扔掉 −3 的前缀，从 −1 重新开始"）；i=2：value = −2 + max(0, −1) = −2。全程最优 best = **−1**。对比跳跃游戏的写法（没有 max(0,·)）：同样的数组它会返回 −3−1−2 里"被迫从头连跳"的结果，这就是两题的分界线。

### 3.2 逆序对计数：上一行的连续段和 + 滑动和

- **Step 1：** 想把长度 n−1 的排列变成长度 n 的：把新的最大数 n 插进去。它比谁都大，插在某个位置就会和它右边的每个数各形成一个逆序对；插法有 n 种，新增逆序对数是 0..n−1。
- **Step 2：** 所以 dp[n][inv] = dp[n−1][inv] + dp[n−1][inv−1] + … + dp[n−1][inv−(n−1)]（下标越界算 0）——正好是上一行一段**连续区间**的和。
- **Step 3：** 连续区间和用滑动和维护：inv 每右移一格，加进 dp_new_row[inv]、减去滑出长度 n 的那一项。配合滚动数组，空间只要一行。

**手算 `k_inverse_pairs(3,1)`（k=1，dp 数组长度 2）：** 初始行（n=0 的空排列）：dp=[1,0]。长度 1：新窗口宽 1，nxt=[1,0]。长度 2：nxt[0]=dp[0]=1；nxt[1]=dp[0]+dp[1]=1（插 2 新增 0 或 1 个逆序对）→ dp=[1,1]。长度 3：nxt[0]=1；nxt[1]=dp[0]+dp[1]=2（从"0 逆序对+插出 1 个"或"1 逆序对+插出 0 个"两种来路）→ dp=[1,2]。返回 dp[1]=**2**，和枚举 (1,3,2)、(2,1,3) 一致。

## 4. 核心代码：constrained_subset_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def constrained_subset_sum(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque()
    best = float('-inf')
    for i, x in enumerate(nums):
        while q and q[0][0] < i - k:
            q.popleft()
        value = x + max(0, q[0][1] if q else 0)
        best = max(best, value)
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return best
```

### 逐段读懂代码

### 4.1 `constrained_subset_sum(nums, k)`

```python
def constrained_subset_sum(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque()
    best = float('-inf')
    for i, x in enumerate(nums):
        while q and q[0][0] < i - k:
            q.popleft()
        value = x + max(0, q[0][1] if q else 0)
        best = max(best, value)
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return best
```

- `if not nums or k<1: raise ValueError(...)`：空数组或窗口 k<1 都不合法（k=0 意味着选不了任何相邻对），直接抛异常。
- `q=deque(); best=float('-inf')`：q 是单调队列；best 记全局最优。best 初始化成负无穷，保证全负数组也能正确返回最大单元素。
- `while q and q[0][0]<i-k: q.popleft()`：队头是**最老**的候选；下标一旦早于 i−k 就滑出窗口，从队头弹出。
- `value=x+max(0,q[0][1] if q else 0)`：队头是窗口内 dp 值的最大（队列值递减保证）。`max(0, ...)` 是"允许重新开始"：不带前缀、从当前元素单开子序列。队列为空（比如 i=0）时表达式自动取 0。
- `best=max(best,value)`：最优子序列可以结束在任何位置，所以要一路取最大，而不是只看最后一位。
- `while q and q[-1][1]<=value: q.pop()`：从队尾弹出所有值不大于新值的候选——它们更老（更早过期）又不更大，被新状态支配，留着无用。
- `q.append((i,value))`：新状态入队，维持"下标递增、值严格递减"。

### 4.2 `max_result(nums, k)`

```python
def max_result(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque([(0, nums[0])])
    value = nums[0]
    for i in range(1, len(nums)):
        while q[0][0] < i - k:
            q.popleft()
        value = nums[i] + q[0][1]
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return value
```

- 结构与 4.1 几乎相同，差别恰好体现两题语义差异：
- `q=deque([(0,nums[0])]); value=nums[0]`：起点必须是下标 0，它的 dp 值就是 nums[0]，先入队。
- `value=nums[i]+q[0][1]`：**没有** max(0,·)——每个位置（除 0 外）都必须从某个前驱跳过来，链条不能断，也不能"传送重开"。
- `return value`：必须到达最后一个下标，所以答案就是最后一位的 dp 值，不需要 best。
- 过期判断 `while q[0][0]<i-k` 前面不加 `q and`：因为任何 i≥1 的窗口 [i−k, i−1] 至少包含 0（k≥1），队列至少有起点兜底，不会空。

### 4.3 `k_inverse_pairs(n, k)`

```python
def k_inverse_pairs(n, k):
    if min(n, k) < 0 or k > n * (n - 1) // 2:
        return 0
    dp = [1] + [0] * k
    for length in range(1, n + 1):
        nxt = [0] * (k + 1)
        window = 0
        for inv in range(k + 1):
            window += dp[inv]
            if inv >= length:
                window -= dp[inv - length]
            nxt[inv] = window % MOD
        dp = nxt
    return dp[k]
```

- `if min(n,k)<0 or k>n*(n-1)//2: return 0`：负参数，或逆序对超过理论最大值 n(n−1)/2（完全倒序排列）都无解，返回 0。
- `dp=[1]+[0]*k`：第 0 行——空排列有 1 种方式达成 0 个逆序对，其它格是 0。
- `window+=dp[inv]`：窗口向右滑，把新进入区间左端的旧值加进来。
- `if inv>=length: window-=dp[inv-length]`：区间宽度应是 `length`（新增逆序对取值 0..length−1，共 length 项），当右端 inv 超出这个宽度时，把滑出区间的 `dp[inv-length]` 减掉。这样 window 始终精确等于上一行区间 [inv−length+1, inv] 的和。
- `nxt[inv]=window%MOD`：边算边取模。
- `dp=nxt`：滚动数组，只留两行，空间 O(k)。
- `return dp[k]`：n 个数、恰好 k 个逆序对的排列数。

## 5. 分段实现：按顺序运行

**本章接口：** `constrained_subset_sum(nums, k)`、`max_result(nums, k)`、`k_inverse_pairs(n, k)`

### 导入本章使用的库

In [1]:
from collections import deque

### MOD

In [2]:
MOD = 10 ** 9 + 7

### constrained_subset_sum

In [3]:
def constrained_subset_sum(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque()
    best = float('-inf')
    for i, x in enumerate(nums):
        while q and q[0][0] < i - k:
            q.popleft()
        value = x + max(0, q[0][1] if q else 0)
        best = max(best, value)
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return best

### max_result

In [4]:
def max_result(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque([(0, nums[0])])
    value = nums[0]
    for i in range(1, len(nums)):
        while q[0][0] < i - k:
            q.popleft()
        value = nums[i] + q[0][1]
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return value

### k_inverse_pairs

In [5]:
def k_inverse_pairs(n, k):
    if min(n, k) < 0 or k > n * (n - 1) // 2:
        return 0
    dp = [1] + [0] * k
    for length in range(1, n + 1):
        nxt = [0] * (k + 1)
        window = 0
        for inv in range(k + 1):
            window += dp[inv]
            if inv >= length:
                window -= dp[inv - length]
            nxt[inv] = window % MOD
        dp = nxt
    return dp[k]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

nums=[1,-1,-2,4,-7,3]; k=2; q=deque([(0,nums[0])]); rows=[(0,nums[0],list(q))]
for i in range(1,len(nums)):
    while q[0][0]<i-k: q.popleft()
    value=nums[i]+q[0][1]
    while q and q[-1][1]<=value: q.pop()
    q.append((i,value)); rows.append((i,value,list(q)))
show_table(['位置','最优到达分数','单调候选(位置,值)'],rows)

位置,最优到达分数,"单调候选(位置,值)"
0,1,"[(0, 1)]"
1,0,"[(0, 1), (1, 0)]"
2,-1,"[(0, 1), (1, 0), (2, -1)]"
3,4,"[(3, 4)]"
4,-3,"[(3, 4), (4, -3)]"
5,7,"[(5, 7)]"


## 7. 正确性、适用条件与复杂度

队列按下标递增、值严格递减；过期项从头删除，被新状态支配的项从尾删除，每项至多进出一次。将最大元素n插入n−1个数的排列中，会新增0..n−1个逆序对，因此dp[n][k]是旧行一段连续和，滑动更新保持这个区间不变量。

**代价：** 两个窗口DP O(n)时间O(min(n,k))辅助空间；逆序对O(nk)时间O(k)空间。计数模10^9+7。队列优化不自动保留最优路径；恢复路径需要另存前驱O(n)。

### 展开理解

**单调队列为什么删得放心？** 被从队尾弹出的旧候选 j 满足"下标更小、值 ≤ 新值"。对任何将来的位置 i：若 j 和新值都在窗口里，新值至少和 j 一样大，最大值轮不到 j；若新值过期了，比它更老的 j 一定更早过期。两条合起来，j 在任何时刻都不可能是"窗口内唯一的最优"，删掉不损失任何答案。队头弹过期、队尾弹被支配，使得队列永远保持"下标递增、值严格递减"，队头就是窗口最大值。

**max(0,·) 的边界为什么重要？** 约束子序列的合法方案允许从任意元素开头的非空子序列，"加 0"恰好表达了"空前缀"这个隐藏选项，所以全负数组也能选出最大的单个负数。跳跃游戏如果也随手加 0，等于允许跳棋"凭空重生"到任意位置，违反"从 0 出发、到末尾结束"的题意——这正是推导段强调"转移中不能随意加入 0"的原因。

**逆序对转移为什么是连续段和？** 把最大数 n 插进 n−1 个数的排列，插入点右边有几个数就新增几个逆序对，插入点可以选 0..n−1 共 n 个位置，于是"长度 n、逆序 inv"的来路恰好是上一行连续 n 个格子之和。滑动窗口每移动一格做一次加、最多一次减，窗口始终罩住这 n 个格子，这是它维持的不变量。

**复杂度（拿具体数字感受）：** 两个窗口 DP 都是 O(n) 时间（每个元素进队一次、出队至多一次）、辅助空间 O(min(n,k))。n=10^5、k=10^5 时，朴素 O(n·k)≈10^10 次比较要跑几十分钟，单调队列版只要约 20 万次队操作，一瞬间完成。逆序对是 O(n·k) 时间、O(k) 空间：LC629 规模 n=k=1000 时正好 10^6 次加法，毫秒级。最后要记住：队列优化丢掉了"最优从哪个 j 来"，要恢复路径必须另存一个 O(n) 的前驱数组。

## 8. 单元测试：每个用例在检查什么

```python
assert max_result([1, -1, -2, 4, -7, 3], 2) == 7

assert constrained_subset_sum([-3, -1, -2], 2) == -1
```

- 第 1 条：跳跃游戏官方样例，7 的完整手算在 3.1 节，路径 0→1→3→5。
- 第 2 条：全负数组是约束子序列最典型的边界——max(0,·) 让它返回最大单元素 −1；如果实现错写成跳跃游戏那种"必须接前缀"，这里会返回 −3−1−2 连锁的更差值。

第 3 段（双重循环对拍）：

```python
for a in product((-2, 0, 3), repeat=6):
    for k in (1, 2, 4):
        dp = [a[0]]
        sub = []
        for i, x in enumerate(a):
            if i:
                dp.append(x + max(dp[max(0, i - k):i]))
            sub.append(x + max([0] + sub[max(0, i - k):i]))
        assert max_result(a, k) == dp[-1]
        assert constrained_subset_sum(a, k) == max(sub)
```

它用 3^6=729 个含负数、零、正数的数组 × k=1、2、4（k=1 与 k≥len 是两端）做朴素切片 DP 对拍。注意朴素版与优化版的差异都被钉死：跳跃游戏取 `dp[-1]`（终点固定），子序列取 `max(sub)`（终点任意）且转移带 `max([0]+...)`（允许重开）。

第 4 段（排列暴力）：对 n=0..7，枚举全部 n! 个排列，用双重循环数逆序对，统计每个逆序数的排列个数，再对 k 从 0 到最大逆序数+1 逐个断言 `k_inverse_pairs(n,k)` 等于真实个数（多出来的 k 应回 0，覆盖 `k>n*(n-1)//2` 的短路分支）。

In [7]:
assert max_result([1, -1, -2, 4, -7, 3], 2) == 7

assert constrained_subset_sum([-3, -1, -2], 2) == -1

from itertools import product, permutations

for a in product((-2, 0, 3), repeat=6):
    for k in (1, 2, 4):
        dp = [a[0]]
        sub = []
        for i, x in enumerate(a):
            if i:
                dp.append(x + max(dp[max(0, i - k):i]))
            sub.append(x + max([0] + sub[max(0, i - k):i]))
        assert max_result(a, k) == dp[-1]
        assert constrained_subset_sum(a, k) == max(sub)

for n in range(8):
    counts = {}
    for p in permutations(range(n)):
        inv = sum((p[i] > p[j] for i in range(n) for j in range(i + 1, n)))
        counts[inv] = counts.get(inv, 0) + 1
    for k in range(n * (n - 1) // 2 + 2):
        assert k_inverse_pairs(n, k) == counts.get(k, 0)

print('N117: 所有本章断言通过')

N117: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 分别优化时间和空间并报告改变。

**练习 2：** 解释滚动数组后如何恢复路径。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（分别优化时间和空间并报告改变）：** 提示——先各写一个"朴素基准"版本：窗口 DP 用切片扫窗口（O(n·k) 时间、O(n) 空间），逆序对用整张二维表（O(n·k) 空间）。然后分别换上优化件：单调队列把时间降到 O(n)，滚动数组把空间降到 O(k)。报告时用具体规模说清楚，比如 n=2000、k=50 时朴素要 10 万次比较、队列版 4000 次；逆序对 n=k=1000 时二维表 100 万格 vs 两行 2001 格。边界别忘了：k≥n 时窗口覆盖全部、k=1 时窗口只有一格。
- **练习 2（滚动数组后如何恢复路径）：** 提示——滚动数组扔掉了旧行，路径信息要单独留：转移时把"取得最优所用的前驱下标 j"记进一个 `parent[i]` 数组（这个数组不滚动，O(n)），最后从终点沿 parent 一路回跳。用 `[1,-1,-2,4,-7,3]`、k=2 手算：i=1 的前驱是 0，i=3 的前驱是 1，i=5 的前驱是 3，回跳得 5←3←1←0，倒过来就是 0→1→3→5。再想想为什么单调队列版必须额外存这个信息（队列只留了值，支配删除把"谁给的这个值"抹掉了）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

MOD = 10 ** 9 + 7

def constrained_subset_sum(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque()
    best = float('-inf')
    for i, x in enumerate(nums):
        while q and q[0][0] < i - k:
            q.popleft()
        value = x + max(0, q[0][1] if q else 0)
        best = max(best, value)
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return best

def max_result(nums, k):
    if not nums or k < 1:
        raise ValueError('nonempty nums and positive k required')
    q = deque([(0, nums[0])])
    value = nums[0]
    for i in range(1, len(nums)):
        while q[0][0] < i - k:
            q.popleft()
        value = nums[i] + q[0][1]
        while q and q[-1][1] <= value:
            q.pop()
        q.append((i, value))
    return value

def k_inverse_pairs(n, k):
    if min(n, k) < 0 or k > n * (n - 1) // 2:
        return 0
    dp = [1] + [0] * k
    for length in range(1, n + 1):
        nxt = [0] * (k + 1)
        window = 0
        for inv in range(k + 1):
            window += dp[inv]
            if inv >= length:
                window -= dp[inv - length]
            nxt[inv] = window % MOD
        dp = nxt
    return dp[k]

# 示例与回归测试
assert max_result([1, -1, -2, 4, -7, 3], 2) == 7

assert constrained_subset_sum([-3, -1, -2], 2) == -1

from itertools import product, permutations

for a in product((-2, 0, 3), repeat=6):
    for k in (1, 2, 4):
        dp = [a[0]]
        sub = []
        for i, x in enumerate(a):
            if i:
                dp.append(x + max(dp[max(0, i - k):i]))
            sub.append(x + max([0] + sub[max(0, i - k):i]))
        assert max_result(a, k) == dp[-1]
        assert constrained_subset_sum(a, k) == max(sub)

for n in range(8):
    counts = {}
    for p in permutations(range(n)):
        inv = sum((p[i] > p[j] for i in range(n) for j in range(i + 1, n)))
        counts[inv] = counts.get(inv, 0) + 1
    for k in range(n * (n - 1) // 2 + 2):
        assert k_inverse_pairs(n, k) == counts.get(k, 0)

print('N117: 所有本章断言通过')

N117: 所有本章断言通过


## 11. 代表题与迁移

- **1425. Constrained Subsequence Sum**：这就是 `constrained_subset_sum` 的原题——练"窗口 max + 单调队列 + max(0,·) 允许重开"这一整套组合。
- **1696. Jump Game VI**：这就是 `max_result` 的原题——与 1425 共用单调队列，但转移不许加 0，练的是"同一结构、不同约束"的辨析。
- **629. K Inverse Pairs Array**：这就是 `k_inverse_pairs` 的原题——练"把转移认成连续区间和，再用滑动窗口 + 滚动数组压时空"的计数型优化。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。